In [ ]:
!pip install transformers torch

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "gpt2-medium"

print("Loading model, please wait...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
tokenizer.pad_token = tokenizer.eos_token

# Knowledge base for common factual questions
KNOWLEDGE_BASE = {
    "who created python": "Python was created by Guido van Rossum and first released in 1991.",
    "who invented python": "Python was created by Guido van Rossum and first released in 1991.",
    "what is python": "Python is a high-level, interpreted programming language known for its simplicity and readability.",
    "what is artificial intelligence": "Artificial Intelligence (AI) is the simulation of human intelligence by machines, enabling them to learn, reason, and solve problems.",
    "hello": "Hello! Nice to meet you. How can I assist you today?",
    "thank you": "You're welcome! Feel free to ask more questions.",
    "bye": "Goodbye! Have a great day!"
}

def get_kb_response(user_text):
    cleaned = user_text.lower().strip().rstrip("?!")
    for key, answer in KNOWLEDGE_BASE.items():
        if key in cleaned:
            return answer
    return None

def generate_response(user_text, chat_history: list):
    prompt = ""
    for turn in chat_history[-2:]:
        prompt += f"Q: {turn['user']}\nA: {turn['bot']}\n\n"
    prompt += f"Q: {user_text}\nA:"

    inputs = tokenizer.encode(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs.shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response.split("\n")[0].strip()

print("\n" + "=" * 50)
print("Chatbot: Hello! I am your AI assistant.")
print("How can I help you today?")
print("(type 'exit' or 'quit' to end)")
print("=" * 50 + "\n")

chat_history = []

while True:
    user_input = input("You: ").strip()
    if not user_input: continue
    if user_input.lower() in {"exit", "quit"}:
        print("\nChatbot: Goodbye!\n")
        break

    response = get_kb_response(user_input) or generate_response(user_input, chat_history)
    chat_history.append({"user": user_input, "bot": response})
    print(f"\nChatbot: {response}\n")